<a href="https://colab.research.google.com/github/PNicius/IP-Adapter-PCA-Explorer/blob/main/Steering_Stable_Diffusion_with_PCA_and_IP_Adapter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Steering Stable Diffusion with PCA and IP-Adapter**

In [ ]:
# @title Environment Setup
# Install required dependencies
!pip install -q diffusers transformers accelerate ip_adapter umap-learn scikit-learn plotly

In [ ]:
# @title Configuration & Directory Setup

import os

# --- CONFIGURATION ---
DATASET_DIR = "/content/dataset"
OUTPUT_DIR = "/content/output"

# Create necessary directories
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Dataset directory set to: {DATASET_DIR}")
print(f"Output directory set to: {OUTPUT_DIR}")
print("NOTE: Please upload your images to DATASET_DIR before running feature extraction.")

In [ ]:
# @title Initialization

import torch
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from diffusers import StableDiffusionPipeline
from transformers import CLIPVisionModelWithProjection, CLIPImageProcessor
from sklearn.decomposition import PCA
from sklearn.manifold import UMAP
import glob
from PIL import Image
import pickle

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "runwayml/stable-diffusion-v1-5"

# Load Standard Pipeline
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16).to(device)
pipe.load_ip_adapter("huggingface/ip-adapter", subfolder="models", weight_name="ip-adapter_sd15.bin")

# Load Feature Extractor
image_encoder = CLIPVisionModelWithProjection.from_pretrained(
    "huggingface/ip-adapter",
    subfolder="models/image_encoder"
).to(device, dtype=torch.float16)

clip_image_processor = CLIPImageProcessor()

In [ ]:
# @title Feature Extraction & PCA Training

def extract_features(image_paths):
    features = []
    for path in image_paths:
        try:
            img = Image.open(path).convert("RGB")
            img_tensor = clip_image_processor(images=img, return_tensors="pt").pixel_values.to(device, dtype=torch.float16)
            with torch.no_grad():
                image_embeds = image_encoder(img_tensor).image_embeds
            features.append(image_embeds.cpu().numpy().flatten())
        except Exception as e:
            print(f"Skipping {path} due to error: {e}")

    return np.array(features)

# Dynamically load any dataset
image_paths = glob.glob(os.path.join(DATASET_DIR, "*.*"))
if not image_paths:
    print("Warning: No images found in DATASET_DIR!")
else:
    print(f"Extracting features from {len(image_paths)} images...")
    features = extract_features(image_paths)

    print("Training PCA...")
    # Cap components to dataset size to avoid errors on small datasets
    n_components = min(50, len(features))
    pca = PCA(n_components=n_components)
    pca.fit(features)

    # Save PCA model to output directory
    pca_path = os.path.join(OUTPUT_DIR, "pca_model.pkl")
    with open(pca_path, "wb") as f:
        pickle.dump(pca, f)
    print(f"PCA model saved to {pca_path}")

In [ ]:
# @title Analyze Variance (Scree Plot) & PCA Directions

def plot_variance(pca_model):
    plt.figure(figsize=(10, 5))
    plt.plot(np.cumsum(pca_model.explained_variance_ratio_), marker='o', linestyle='--')
    plt.title('Scree Plot: Cumulative Explained Variance')
    plt.xlabel('Number of Components')
    plt.ylabel('Cumulative Explained Variance')
    plt.grid(True)
    plt.savefig(os.path.join(OUTPUT_DIR, "scree_plot.png"))
    plt.show()

def visualize_pca_directions(pca_model, n_components=5):
    n_plot = min(n_components, len(pca_model.components_))
    fig, axes = plt.subplots(1, n_plot, figsize=(15, 3))

    # Handle case where n_plot is 1
    if n_plot == 1: axes = [axes]

    for i in range(n_plot):
        axes[i].hist(pca_model.components_[i], bins=20)
        axes[i].set_title(f'PCA Dir {i+1}')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "pca_directions.png"))
    plt.show()

if 'pca' in locals():
    plot_variance(pca)
    visualize_pca_directions(pca)

In [ ]:
# @title Generate & Show 3D UMAP Plot

def generate_umap_3d(feature_matrix):
    if len(feature_matrix) < 3:
        print("Not enough data points for 3D UMAP. Need at least 3 images.")
        return

    umap_3d = UMAP(n_components=3, random_state=42)
    proj_3d = umap_3d.fit_transform(feature_matrix)

    fig = px.scatter_3d(
        x=proj_3d[:, 0], y=proj_3d[:, 1], z=proj_3d[:, 2],
        title='3D UMAP Projection of Image Features',
        opacity=0.7
    )

    html_path = os.path.join(OUTPUT_DIR, "umap_3d.html")
    fig.write_html(html_path)
    print(f"3D UMAP plot saved to {html_path}")
    fig.show()

if 'features' in locals():
    generate_umap_3d(features)

In [ ]:
# @title Run Experiment

def run_experiment(pca_model, component_idx=0, scale=5.0, prompt="A high quality photo"):
    print(f"Running experiment: Modifying PCA component {component_idx} with scale {scale}")

    # 1. Base embedding (mean feature)
    mean_feature = pca_model.mean_

    # 2. Shift the embedding along the selected principal component
    modified_feature = mean_feature + (scale * pca_model.components_[component_idx])

    # 3. Preserve input conditioning structure for IP-Adapter
    ip_embeds = torch.tensor(modified_feature, dtype=torch.float16, device=device).unsqueeze(0)

    # Generate
    result = pipe(
        prompt=prompt,
        ip_adapter_image_embeds=[ip_embeds],
        num_inference_steps=30,
        guidance_scale=7.5
    ).images[0]

    # Save the output locally
    exp_dir = os.path.join(OUTPUT_DIR, "experiment_results")
    os.makedirs(exp_dir, exist_ok=True)
    out_path = os.path.join(exp_dir, f"exp_comp{component_idx}_scale{scale}.png")
    result.save(out_path)

    # Display inline
    plt.figure(figsize=(6, 6))
    plt.imshow(result)
    plt.axis('off')
    plt.title(f"Component {component_idx} | Scale {scale}")
    plt.show()

# To test the generation, uncomment the line below:
# run_experiment(pca, component_idx=0, scale=3.0, prompt="A creative portrait")